In [ ]:
from pathlib import Path
import base64, hashlib, io, json, shutil, subprocess, sys, zipfile

work=Path('/root/shared-nvme/pha-intent-qwen4b')
work.mkdir(parents=True, exist_ok=True)
browser_dir=Path.cwd()
print(subprocess.check_output(['nvidia-smi','-L'],text=True).strip())
payload=base64.b64decode('...')
assert hashlib.sha256(payload).hexdigest() == 'f05de1d18b21fb4595d2d696e5ffca053b5f6335f048f1416429df610c1cd453'
with zipfile.ZipFile(io.BytesIO(payload)) as archive:
    archive.extractall(work)
print('Project and synthetic data staged in',work)
driver=r'''
from pathlib import Path
import os, shutil, subprocess, sys

def run(args):
    subprocess.run(args, check=True)

packages=['transformers==4.57.3','peft==0.18.0','accelerate==1.12.0','modelscope==1.32.0','tokenizers==0.22.1','huggingface-hub==0.36.2']
pip=[sys.executable,'-m','pip','install','--no-deps','--no-compile','--target','/tmp/pha-qwen4b-pydeps','--disable-pip-version-check','--retries','2']
try:
    run([*pip,'-i','https://pypi.tuna.tsinghua.edu.cn/simple',*packages])
except subprocess.CalledProcessError:
    run([*pip,*packages])
deps=str(Path('/tmp/pha-qwen4b-pydeps').resolve())
sys.path.insert(0,deps)
os.environ['PYTHONPATH']=deps+os.pathsep+os.environ.get('PYTHONPATH','')
import torch, peft, transformers
print('CUDA environment',torch.__version__,peft.__version__,transformers.__version__,flush=True)
from modelscope import snapshot_download
snapshot_download('Qwen/Qwen3-4B-Instruct-2507',local_dir='weights')
run([sys.executable,'-u','-m','experiments.intent_experiment.train_cuda','--model','weights','--out','run','--epochs','2','--batch','4'])
delivery=Path(sys.argv[1])
shutil.make_archive(str(delivery),'zip',root_dir='.',base_dir='run')
print('Report package ready', str(delivery) + '.zip',flush=True)
'''
(work/'run_cloud.py').write_text(driver,encoding='utf-8')
with (work/'training.log').open('ab') as log:
    process=subprocess.Popen(['timeout','65m','/usr/bin/python3','-u',str(work/'run_cloud.py'),str(browser_dir/'pha_qwen4b_results')],cwd=work,stdin=subprocess.DEVNULL,stdout=log,stderr=subprocess.STDOUT,start_new_session=True)
print('Cloud job launched with PID',process.pid)


In [ ]:
from pathlib import Path
log=Path('/root/shared-nvme/pha-intent-qwen4b/training.log')
print(log.read_text(encoding='utf-8',errors='replace')[-4500:] if log.exists() else 'The cloud job has not started.')
